# Stats Notebook #1 — Is a Difference *Real*?

You discovered something important on your own: the country comparison *looked*
like a big difference (Italy 10.24 vs Japan 5.47), but the country labels were
assigned **at random** — so any difference must be noise. Today you learn the
tool that tells signal from noise: the **statistical test**.

**Same rules as before:** Read → Run (`Shift+Enter`) → Modify (do the YOUR TURN
tasks, predict first) → Explain (in your own words).

The one big idea:
> **A difference you can SEE is not the same as a difference that is REAL.**
> A statistical test estimates how likely a difference this big could happen by
> pure chance. If "by chance" is very unlikely, we call it real (significant).

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats   # scipy = scientific python; has the stats tests

df = pd.read_csv("example_gut_samples.csv")
print("Loaded", df.shape[0], "samples")

Loaded 24 samples


## 2. The p-value — what it actually means

The **p-value** answers one question:

> *"IF there were truly no difference between the groups, how often would I see a
> gap at least this big, just by random luck?"*

- **Small p (e.g. 0.001)** → a gap this big would almost never happen by chance → the difference is probably **real**.
- **Large p (e.g. 0.40)** → a gap this big happens by chance all the time → we **can't claim** a real difference.

The traditional cutoff is **p < 0.05** = "statistically significant." (Keep that
number slightly suspicious — you'll see why below.)

## 3. Test a REAL difference: low_mood vs control

We compare *Faecalibacterium* between the two mood groups using a **t-test**
(`ttest_ind` = "independent-samples t-test"). It returns two numbers: a
`t` statistic (size of the gap, in noise units) and the `p` value.

In [3]:
low = df[df["group"]=="low_mood"]["Faecalibacterium"]
ctl = df[df["group"]=="control"]["Faecalibacterium"]

t, p = stats.ttest_ind(low, ctl)
print(f"low_mood mean = {low.mean():.2f}")
print(f"control  mean = {ctl.mean():.2f}")
print(f"t = {t:.2f},  p = {p:.4f}")

low_mood mean = 5.65
control  mean = 11.29
t = -6.53,  p = 0.0000


### What you should see
p is essentially 0 (way below 0.05). Translation: a gap this big would basically
never happen by chance → **this difference is real.** (It is — I built it in.)

### EXPLAIN #1

*In your own words: what does the p-value here tell you about the low_mood vs control difference?*

> your answer here

## 4. Now test the FAKE difference: Italy vs Japan

Same test, but on `country` — which I assigned **at random**. There is no real
country effect. Let's see what the test says.

In [4]:
italy = df[df["country"]=="Italy"]["Faecalibacterium"]
japan = df[df["country"]=="Japan"]["Faecalibacterium"]

t, p = stats.ttest_ind(italy, japan)
print(f"Italy mean = {italy.mean():.2f}  (n={len(italy)})")
print(f"Japan mean = {japan.mean():.2f}  (n={len(japan)})")
print(f"t = {t:.2f},  p = {p:.4f}")

Italy mean = 10.24  (n=8)
Japan mean = 5.47  (n=6)
t = 2.83,  p = 0.0151


### 🚨 The lesson — read this carefully
You'll likely get **p ≈ 0.015 — below 0.05, so it "passes" as significant.**
But we KNOW there's no real country effect (random labels). So this is a
**FALSE POSITIVE**: noise that got lucky and crossed the line.

This is not a bug — it's the nature of the 0.05 cutoff. If "significant" means
"happens <5% of the time by chance," then **about 1 in 20 pure-noise tests will
be 'significant' anyway.** Test enough things and you're guaranteed false hits.

Why it looked convincing: tiny samples (only 6–8 people per country). Small
groups make random noise look like real patterns.

### EXPLAIN #2

*Why is the Italy vs Japan 'significant' result misleading? What would make you trust a difference more?*

> your answer here

## 5. See the false-positive problem with your own eyes

Let's prove "1 in 20 by chance" is real. We'll invent a **completely random**
label (a coin flip unrelated to anything), split the data by it, test it, and
repeat 1000 times. Then count how often we get p < 0.05.

In [5]:
rng = np.random.default_rng(0)
vals = df["Faecalibacterium"].values
false_hits = 0
n_tests = 1000

for i in range(n_tests):
    fake_label = rng.integers(0, 2, size=len(vals))   # random 0/1 per person
    g0 = vals[fake_label==0]
    g1 = vals[fake_label==1]
    _, p = stats.ttest_ind(g0, g1)
    if p < 0.05:
        false_hits += 1

print(f"Out of {n_tests} tests on RANDOM labels,")
print(f"{false_hits} came out 'significant' (p<0.05).")
print(f"That's {100*false_hits/n_tests:.1f}% — in the ballpark of the expected 5%.")

Out of 1000 tests on RANDOM labels,
34 came out 'significant' (p<0.05).
That's 3.4% — in the ballpark of the expected 5%.


### What you should see
Roughly **3–6% of 1000** random, meaningless splits come out
"significant" (the exact number wobbles run-to-run because we only have 24
people — small data is noisy, which is itself the point). That's the false-positive rate baked into p < 0.05. **This is why
you can't just test many things and trust whatever crosses 0.05.**

The fix — testing many taxa at once and correcting for it (*multiple-testing
correction*) — is the next stats notebook. This is quiz question #16 come to
life.

### EXPLAIN #3

*In one sentence: why does testing lots of things make false positives a big problem?*

> your answer here

## 6. 🔧 YOUR TURN — test a bacterium yourself

Pick a different bacterium (e.g. `Bifidobacterium`, `Bacteroides`, `Prevotella`)
and test low_mood vs control. **Predict first:** from your earlier plots, which
ones do you expect to be significant (small p) vs not?

In [6]:
# Change "Bifidobacterium" to any column, predict the p-value size, then run:
col = "Bifidobacterium"
a = df[df["group"]=="low_mood"][col]
b = df[df["group"]=="control"][col]
t, p = stats.ttest_ind(a, b)
print(f"{col}: low_mood={a.mean():.2f}, control={b.mean():.2f}, p={p:.4f}")

Bifidobacterium: low_mood=2.51, control=5.26, p=0.0054


### EXPLAIN #4

*Which bacteria were significant (p<0.05) between mood groups? Did it match your prediction from the plots?*

> your yes, p = 0.054

## Done — you can now tell signal from noise

You learned:
1. **p-value** = probability a difference this big is just chance.
2. **p < 0.05** = the conventional "significant" line — but it's not magic.
3. **False positives are real** — ~5% of pure-noise tests pass, so testing many
   things guarantees some false hits.
4. You **tested it yourself** and reproduced the 5% false-positive rate.

**Next:** tell me your EXPLAIN answers or say "what's next" — the next step is
either multiple-testing correction (finishing this thread) or moving to real
public data. Save your notebook: File → Save (Cmd+S).